In [ ]:
# Databricks 노트북: Binance Kline REST API → Bronze 적재 (4h 봉 전용)
import time, json, random, datetime as dt, hashlib, requests
from typing import List, Dict, Tuple, Optional
from pyspark.sql import Row
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, LongType

# Delta Lake 소파일 자동 병합 및 백그라운드 컴팩션
spark.conf.set("spark.databricks.delta.optimizeWrite","true")
spark.conf.set("spark.databricks.delta.autoCompact","true")
spark.sql("SET spark.sql.session.timeZone=UTC")  # 모든 시간은 UTC 기준

# ============= 실행/프로젝트 설정 =============
MODE           = "once"                         # once | poll | forever | backfill
SYMBOLS        = ["BTCUSDT","ETHUSDT","SOLUSDT"]
INTERVALS      = ["4h"]                         # 4시간 봉만 수집
LIMIT_ONCE     = 1000                           # 1회 수집 최대 캔들 수
POLL_SECONDS   = 60                             # 폴링 간격(초)
MAX_POLLS      = 10                             # poll 모드 최대 반복 횟수
BACKFILL_DAYS  = 200                            # MA200 안정화에 필요한 과거 데이터 기간

CATALOG = "demo_catalog"
SCHEMA  = "demo_schema"
TABLE       = f"{CATALOG}.{SCHEMA}.bronze_charts"        # 원본 Append-Only 테이블
STATE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_ingest_state"  # 심볼별 마지막 open_time(ms) 체크포인트

BASE_URL = "https://api.binance.com"
KLINES   = "/api/v3/klines"
LIMIT_DEFAULT = 1000

# ============= 테이블 준비 (자가 부트스트래핑) =============
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOG}.{SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
  source            STRING,
  symbol            STRING,
  interval          STRING,
  event_time        TIMESTAMP,  -- open_time(UTC)
  ingest_time       TIMESTAMP,  -- 적재 시각(UTC)
  unique_key        STRING,     -- "symbol|interval|open_time(ms)"
  raw_json          STRING,     -- 원본 배열 JSON 문자열 (Audit Trail)
  api_endpoint      STRING,     -- 호출한 REST 엔드포인트 경로
  api_params_hash   STRING,     -- 요청 파라미터 SHA256 해시 (재현성 확보)
  dt                DATE        -- 파티션 컬럼 (Partition Pruning)
) USING DELTA
PARTITIONED BY (dt)
TBLPROPERTIES (
  'delta.logRetentionDuration'         = 'interval 7 days',
  'delta.deletedFileRetentionDuration' = 'interval 7 days'
)
""")

# 상태 테이블: 심볼×인터벌별 마지막 수집 위치 저장 → 재시작 시 중복 수집 방지
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
  symbol            STRING,
  interval          STRING,
  last_open_time_ms LONG,      -- 마지막 수집된 open_time(밀리초)
  updated_at        TIMESTAMP
) USING DELTA
""")

# ============= 유틸리티 함수 =============
def _params_hash(params: Dict) -> str:
    """요청 파라미터를 정렬 후 SHA256 해시 → 동일 요청 식별용"""
    raw = json.dumps(params, sort_keys=True, separators=(",",":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def _to_ms(ts: dt.datetime) -> int:
    """datetime → 밀리초 타임스탬프 변환"""
    if ts.tzinfo is None:
        ts = ts.replace(tzinfo=dt.timezone.utc)
    return int(ts.timestamp() * 1000)

def _from_ms(ms: int) -> dt.datetime:
    """밀리초 타임스탬프 → UTC datetime 변환"""
    return dt.datetime.fromtimestamp(ms/1000, tz=dt.timezone.utc)

def binance_klines(symbol: str, interval: str, start_ms: Optional[int]=None,
                   end_ms: Optional[int]=None, limit: int=LIMIT_DEFAULT,
                   max_retries: int=5) -> Tuple[List[list], Dict[str,str]]:
    """
    Binance Kline REST API 호출.
    - 429(Rate Limit) 발생 시 Exponential Backoff + Jitter 재시도
    - 그 외 오류는 즉시 실패 처리
    """
    url = BASE_URL + KLINES
    q = {"symbol": symbol, "interval": interval, "limit": limit}
    if start_ms is not None: q["startTime"] = start_ms
    if end_ms   is not None: q["endTime"]   = end_ms

    delay, last_headers = 1, {}
    for _ in range(max_retries):
        r = requests.get(url, params=q, timeout=30)
        last_headers = {k:v for k,v in r.headers.items()}
        if r.status_code == 200:
            return r.json(), last_headers
        if r.status_code == 429:
            # Rate Limit 초과: Exponential Backoff + 무작위 Jitter로 버스트 회피
            time.sleep(delay + random.uniform(0, 0.3))
            delay = min(delay * 2, 16)
            continue
        time.sleep(1 + random.uniform(0, 0.3))
    r.raise_for_status()
    return [], last_headers

def _get_last_state(symbol: str, interval: str) -> Optional[int]:
    """상태 테이블에서 마지막 수집 위치(open_time ms) 조회"""
    df = spark.sql(f"""
      SELECT last_open_time_ms
      FROM {STATE_TABLE}
      WHERE symbol = '{symbol}' AND interval = '{interval}'
      ORDER BY updated_at DESC
      LIMIT 1
    """)
    rows = df.collect()
    return rows[0][0] if rows else None

def _upsert_state(symbol: str, interval: str, last_ms: int):
    """상태 테이블 업서트 → 재시작 시 이 위치부터 재개"""
    now = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    spark.sql(f"""
      MERGE INTO {STATE_TABLE} t
      USING (SELECT '{symbol}' AS symbol, '{interval}' AS interval,
                    {last_ms} AS last_open_time_ms, TIMESTAMP('{now}') AS updated_at) s
      ON t.symbol = s.symbol AND t.interval = s.interval
      WHEN MATCHED THEN UPDATE SET last_open_time_ms = s.last_open_time_ms, updated_at = s.updated_at
      WHEN NOT MATCHED THEN INSERT (symbol, interval, last_open_time_ms, updated_at)
      VALUES (s.symbol, s.interval, s.last_open_time_ms, s.updated_at)
    """)

def _append_to_bronze(symbol: str, interval: str, rows: List[list], endpoint: str, params: Dict):
    """캔들 데이터를 Bronze 테이블에 Append (unique_key 기준 중복 제거)"""
    if not rows: return 0, None
    param_hash = _params_hash(params)
    now = dt.datetime.now(dt.timezone.utc)
    now_s = now.strftime("%Y-%m-%d %H:%M:%S")

    recs, max_open_ms = [], None
    for item in rows:
        open_ms = int(item[0])
        event_time = _from_ms(open_ms)
        unique = f"{symbol}|{interval}|{open_ms}"
        recs.append({
          "source":          "binance.spot.klines",
          "symbol":          symbol,
          "interval":        interval,
          "event_time":      event_time.strftime("%Y-%m-%d %H:%M:%S"),
          "ingest_time":     now_s,
          "unique_key":      unique,
          "raw_json":        json.dumps(item, separators=(",",":")),
          "api_endpoint":    endpoint,
          "api_params_hash": param_hash,
          "dt":              event_time.date().isoformat()
        })
        if (max_open_ms is None) or (open_ms > max_open_ms):
            max_open_ms = open_ms

    schema = StructType([
        StructField("source",           StringType(), True),
        StructField("symbol",           StringType(), True),
        StructField("interval",         StringType(), True),
        StructField("event_time",       StringType(), True),
        StructField("ingest_time",      StringType(), True),
        StructField("unique_key",       StringType(), True),
        StructField("raw_json",         StringType(), True),
        StructField("api_endpoint",     StringType(), True),
        StructField("api_params_hash",  StringType(), True),
        StructField("dt",               StringType(), True),
    ])
    df = (spark.createDataFrame([Row(**r) for r in recs], schema)
            .withColumn("event_time",  to_timestamp(col("event_time")))
            .withColumn("ingest_time", to_timestamp(col("ingest_time")))
            .withColumn("dt",          col("dt").cast("date"))
            .dropDuplicates(["unique_key"])
            .repartition("dt"))

    count = df.count()
    df.writeTo(TABLE).append()
    return count, max_open_ms

# ============= 모드별 수집 로직 =============
def backfill_symbol(symbol: str, interval: str, days: int = BACKFILL_DAYS, limit: int = LIMIT_DEFAULT):
    """
    과거 데이터 전체 백필.
    - 상태 테이블에 마지막 위치가 있으면 그 다음부터 재개 (중복 수집 방지)
    - 4h 봉 기준: 15일 창으로 분할 → API 호출 수 최소화
    """
    now_utc   = dt.datetime.now(dt.timezone.utc)
    start_utc = now_utc - dt.timedelta(days=days)
    last_ms = _get_last_state(symbol, interval)
    if last_ms:
        start_utc = _from_ms(last_ms) + dt.timedelta(milliseconds=1)

    start_ms, end_ms = _to_ms(start_utc), _to_ms(now_utc)
    total = 0
    cursor_ms = start_ms
    print(f"[백필] {symbol} {interval}: {_from_ms(start_ms)} → {_from_ms(end_ms)}")

    # 4h 봉은 15일 창으로 분할해도 호출 수가 적음 (15일 = 90캔들)
    step_ms = 1000 * 60 * 60 * 24 * 15
    while cursor_ms < end_ms:
        batch_end = min(end_ms, cursor_ms + step_ms)
        params = {"symbol": symbol, "interval": interval, "startTime": cursor_ms, "endTime": batch_end, "limit": limit}
        rows, _ = binance_klines(symbol, interval, start_ms=cursor_ms, end_ms=batch_end, limit=limit)
        cnt, max_open_ms = _append_to_bronze(symbol, interval, rows, KLINES, params)
        total += cnt
        if max_open_ms is None:
            cursor_ms = batch_end + 1
        else:
            cursor_ms = max_open_ms + 1
            _upsert_state(symbol, interval, max_open_ms)
        time.sleep(0.2)
    print(f"[백필 완료] {symbol} {interval}: 총 {total}행 적재")

def poll_once(symbol: str, interval: str, limit: int = 500):
    """최신 캔들 1회 수집 → Bronze 적재 → 상태 테이블 갱신"""
    params = {"symbol": symbol, "interval": interval, "limit": limit}
    rows, headers = binance_klines(symbol, interval, limit=limit)
    cnt, max_open_ms = _append_to_bronze(symbol, interval, rows, KLINES, params)
    if max_open_ms is not None:
        _upsert_state(symbol, interval, max_open_ms)
    # API 가중치 모니터링: 1200 초과 시 Rate Limit 위험
    used_weight = headers.get("X-MBX-USED-WEIGHT-1m") or headers.get("X-MBX-USED-WEIGHT")
    print(f"[수집] {symbol} {interval}: +{cnt}행, 마지막open_ms={max_open_ms}, API가중치={used_weight}")

# ============= MAIN =============
if MODE == "backfill":
    for sym in SYMBOLS:
        backfill_symbol(sym, "4h", days=BACKFILL_DAYS, limit=LIMIT_DEFAULT)
    dbutils.notebook.exit("백필 완료")
elif MODE == "poll":
    for i in range(MAX_POLLS):
        for sym in SYMBOLS:
            poll_once(sym, "4h", limit=LIMIT_ONCE)
        time.sleep(POLL_SECONDS)
    dbutils.notebook.exit("폴링 완료")
elif MODE == "forever":
    print(f"[라이브] {POLL_SECONDS}초 간격 무한 폴링 시작")
    while True:
        try:
            for sym in SYMBOLS:
                poll_once(sym, "4h", limit=LIMIT_ONCE)
        except Exception as e:
            print(f"[경고] {e}")
            time.sleep(5)
        time.sleep(POLL_SECONDS)
else:  # once
    for sym in SYMBOLS:
        poll_once(sym, "4h", limit=LIMIT_ONCE)
    dbutils.notebook.exit("1회 수집 완료")